# Intalação da Biblioteca **Pyspark**

In [1]:
!pip install pyspark

# Intalação da Biblioteca **FindaSpark**

In [3]:
!pip install findspark

In [18]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [19]:
df = spark.sql("""select 'spark' as hello """)
df.show()

+-----+
|hello|
+-----+
|spark|
+-----+



# Importando **Spark** libraries

In [52]:
from pyspark.sql import Row, dataframe
from pyspark.sql.types import StringType, StructType, StructField, IntegralType
from pyspark.sql.functions import col, expr, lit, substring, concat, concat_ws, when, coalesce, DataFrame
from pyspark.sql import functions as F
from functools import reduce

# Data Manipulation Unsing Spark

In [27]:
df = spark.read.csv('banklist.csv', sep = ',', inferSchema = True, header= True)

print('df.count   :', df.count())
print('df.col ct  :', len(df.columns))
print('df.columns :', df.columns)

df.count   : 561
df.col ct  : 6
df.columns : ['Bank Name', 'City', 'ST', 'CERT', 'Acquiring Institution', 'Closing Date']


# Using SQL in PySpark

In [30]:
from os import truncate
df.createOrReplaceTempView('banklist')

df_check = spark.sql('''select `City`, `ST`, `CERT`, `Closing Date` from banklist''')
df_check.show(4, truncate = False)

+-------------+---+-----+------------+
|City         |ST |CERT |Closing Date|
+-------------+---+-----+------------+
|Barboursville|WV |14361|3-Apr-20    |
|Ericson      |NE |18265|14-Feb-20   |
|Newark       |NJ |21111|1-Nov-19    |
|Maumee       |OH |58317|25-Oct-19   |
+-------------+---+-----+------------+
only showing top 4 rows



# DataFrame Basic Operations

In [32]:
df.describe().show()

+-------+--------------------+-------+----+-----------------+---------------------+------------+
|summary|           Bank Name|   City|  ST|             CERT|Acquiring Institution|Closing Date|
+-------+--------------------+-------+----+-----------------+---------------------+------------+
|  count|                 561|    561| 561|              561|                  561|         561|
|   mean|                NULL|   NULL|NULL|31685.68449197861|                 NULL|        NULL|
| stddev|                NULL|   NULL|NULL|16446.65659309965|                 NULL|        NULL|
|    min|1st American Stat...|Acworth|  AL|               91|      1st United Bank|    1-Aug-08|
|    max|               ebank|Wyoming|  WY|            58701|  Your Community Bank|    9-Sep-11|
+-------+--------------------+-------+----+-----------------+---------------------+------------+



In [33]:
df.describe('City','ST').show()

+-------+-------+----+
|summary|   City|  ST|
+-------+-------+----+
|  count|    561| 561|
|   mean|   NULL|NULL|
| stddev|   NULL|NULL|
|    min|Acworth|  AL|
|    max|Wyoming|  WY|
+-------+-------+----+



# Count, Columns and Schema

In [39]:
print('Total de Linhas: ', df.count())
print('Colunas: ', df.columns)
print('Tipo de dados: ', df.dtypes)
print('df schema', df.schema)

Total de Linhas:  561
Colunas:  ['Bank Name', 'City', 'ST', 'CERT', 'Acquiring Institution', 'Closing Date']
Tipo de dados:  [('Bank Name', 'string'), ('City', 'string'), ('ST', 'string'), ('CERT', 'int'), ('Acquiring Institution', 'string'), ('Closing Date', 'string')]
df schema StructType([StructField('Bank Name', StringType(), True), StructField('City', StringType(), True), StructField('ST', StringType(), True), StructField('CERT', IntegerType(), True), StructField('Acquiring Institution', StringType(), True), StructField('Closing Date', StringType(), True)])


In [40]:
df.printSchema()

root
 |-- Bank Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- ST: string (nullable = true)
 |-- CERT: integer (nullable = true)
 |-- Acquiring Institution: string (nullable = true)
 |-- Closing Date: string (nullable = true)



# Remove Duplicates

In [41]:
df = df.dropDuplicates()

print('df.count   :', df.count())
print('df.col ct  :', len(df.columns))
print('df.columns :', df.columns)

df.count   : 561
df.col ct  : 6
df.columns : ['Bank Name', 'City', 'ST', 'CERT', 'Acquiring Institution', 'Closing Date']


# Select Specific Columns

In [43]:
df2 = df.select(*['Bank Name', 'City'])
df2.show(2)

+--------------------+--------+
|           Bank Name|    City|
+--------------------+--------+
| First Bank of Idaho| Ketchum|
|Amcore Bank, Nati...|Rockford|
+--------------------+--------+
only showing top 2 rows



# Select Multiple Columns

In [45]:
col_l = list(set(df.columns) - set(['Bank Name', 'City']))
df2 = df.select(*col_l)
df2.show(10)

+-----+---------------------+---+------------+
| CERT|Acquiring Institution| ST|Closing Date|
+-----+---------------------+---+------------+
|34396|      U.S. Bank, N.A.| ID|   24-Apr-09|
| 3735|          Harris N.A.| IL|   23-Apr-10|
|22868| First-Citizens Ba...| WA|   11-Sep-09|
| 9873|         Herring Bank| OK|   31-Jul-09|
|58399| Enterprise Bank &...| AZ|   11-Dec-09|
|34369|       Level One Bank| MI|   24-Apr-09|
|32284| United Fidelity B...| OH|   23-May-14|
|33883| The Huntington Na...| MI|   30-Mar-12|
|19797|   Bank of the Ozarks| GA|   29-Apr-11|
|58087| First California ...| CA|    5-Nov-10|
+-----+---------------------+---+------------+
only showing top 10 rows



# Rename Columns

In [46]:
df2 = df \
  .withColumnRenamed('Bank Name','bank name') \
  .withColumnRenamed('Acquiring Institution', 'acquiring institution') \
  .withColumnRenamed('Closing Date', 'closing date') \
  .withColumnRenamed('ST', 'state') \
  .withColumnRenamed('CERT', 'cert')

df2.show(2)

+--------------------+--------+-----+-----+---------------------+------------+
|           bank name|    City|state| cert|acquiring institution|closing date|
+--------------------+--------+-----+-----+---------------------+------------+
| First Bank of Idaho| Ketchum|   ID|34396|      U.S. Bank, N.A.|   24-Apr-09|
|Amcore Bank, Nati...|Rockford|   IL| 3735|          Harris N.A.|   23-Apr-10|
+--------------------+--------+-----+-----+---------------------+------------+
only showing top 2 rows



# Add Columns

In [47]:
df2 = df.withColumn('state', col('ST'))
df2.show()

+--------------------+----------------+---+-----+---------------------+------------+-----+
|           Bank Name|            City| ST| CERT|Acquiring Institution|Closing Date|state|
+--------------------+----------------+---+-----+---------------------+------------+-----+
| First Bank of Idaho|         Ketchum| ID|34396|      U.S. Bank, N.A.|   24-Apr-09|   ID|
|Amcore Bank, Nati...|        Rockford| IL| 3735|          Harris N.A.|   23-Apr-10|   IL|
|        Venture Bank|           Lacey| WA|22868| First-Citizens Ba...|   11-Sep-09|   WA|
|First State Bank ...|           Altus| OK| 9873|         Herring Bank|   31-Jul-09|   OK|
|Valley Capital Ba...|            Mesa| AZ|58399| Enterprise Bank &...|   11-Dec-09|   AZ|
|Michigan Heritage...|Farmington Hills| MI|34369|       Level One Bank|   24-Apr-09|   MI|
|Columbia Savings ...|      Cincinnati| OH|32284| United Fidelity B...|   23-May-14|   OH|
|       Fidelity Bank|        Dearborn| MI|33883| The Huntington Na...|   30-Mar-12|   MI|

# Add Constant Column

In [48]:
df2 = df.withColumn('country', lit('US'))
df2.show()

+--------------------+----------------+---+-----+---------------------+------------+-------+
|           Bank Name|            City| ST| CERT|Acquiring Institution|Closing Date|country|
+--------------------+----------------+---+-----+---------------------+------------+-------+
| First Bank of Idaho|         Ketchum| ID|34396|      U.S. Bank, N.A.|   24-Apr-09|     US|
|Amcore Bank, Nati...|        Rockford| IL| 3735|          Harris N.A.|   23-Apr-10|     US|
|        Venture Bank|           Lacey| WA|22868| First-Citizens Ba...|   11-Sep-09|     US|
|First State Bank ...|           Altus| OK| 9873|         Herring Bank|   31-Jul-09|     US|
|Valley Capital Ba...|            Mesa| AZ|58399| Enterprise Bank &...|   11-Dec-09|     US|
|Michigan Heritage...|Farmington Hills| MI|34369|       Level One Bank|   24-Apr-09|     US|
|Columbia Savings ...|      Cincinnati| OH|32284| United Fidelity B...|   23-May-14|     US|
|       Fidelity Bank|        Dearborn| MI|33883| The Huntington Na...

# Drop Column

In [49]:
df2 = df.drop('CERT')
df2.show()

+--------------------+----------------+---+---------------------+------------+
|           Bank Name|            City| ST|Acquiring Institution|Closing Date|
+--------------------+----------------+---+---------------------+------------+
| First Bank of Idaho|         Ketchum| ID|      U.S. Bank, N.A.|   24-Apr-09|
|Amcore Bank, Nati...|        Rockford| IL|          Harris N.A.|   23-Apr-10|
|        Venture Bank|           Lacey| WA| First-Citizens Ba...|   11-Sep-09|
|First State Bank ...|           Altus| OK|         Herring Bank|   31-Jul-09|
|Valley Capital Ba...|            Mesa| AZ| Enterprise Bank &...|   11-Dec-09|
|Michigan Heritage...|Farmington Hills| MI|       Level One Bank|   24-Apr-09|
|Columbia Savings ...|      Cincinnati| OH| United Fidelity B...|   23-May-14|
|       Fidelity Bank|        Dearborn| MI| The Huntington Na...|   30-Mar-12|
|The Park Avenue Bank|        Valdosta| GA|   Bank of the Ozarks|   29-Apr-11|
|Western Commercia...|  Woodland Hills| CA| First Ca

# Drop Multiple Columns

In [50]:
df2 = df.drop(*['CERT', 'ST'])
df2.show()

+--------------------+----------------+---------------------+------------+
|           Bank Name|            City|Acquiring Institution|Closing Date|
+--------------------+----------------+---------------------+------------+
| First Bank of Idaho|         Ketchum|      U.S. Bank, N.A.|   24-Apr-09|
|Amcore Bank, Nati...|        Rockford|          Harris N.A.|   23-Apr-10|
|        Venture Bank|           Lacey| First-Citizens Ba...|   11-Sep-09|
|First State Bank ...|           Altus|         Herring Bank|   31-Jul-09|
|Valley Capital Ba...|            Mesa| Enterprise Bank &...|   11-Dec-09|
|Michigan Heritage...|Farmington Hills|       Level One Bank|   24-Apr-09|
|Columbia Savings ...|      Cincinnati| United Fidelity B...|   23-May-14|
|       Fidelity Bank|        Dearborn| The Huntington Na...|   30-Mar-12|
|The Park Avenue Bank|        Valdosta|   Bank of the Ozarks|   29-Apr-11|
|Western Commercia...|  Woodland Hills| First California ...|    5-Nov-10|
|        Syringa Bank|   

In [53]:
df2 = reduce(DataFrame.drop, ['CERT', 'ST'], df)
df2.show(2)

+--------------------+--------+---------------------+------------+
|           Bank Name|    City|Acquiring Institution|Closing Date|
+--------------------+--------+---------------------+------------+
| First Bank of Idaho| Ketchum|      U.S. Bank, N.A.|   24-Apr-09|
|Amcore Bank, Nati...|Rockford|          Harris N.A.|   23-Apr-10|
+--------------------+--------+---------------------+------------+
only showing top 2 rows



# Filter Data

In [56]:
df2 = df.where(df['ST'] == 'NE')

df3 = df.where(df['CERT'].between('1000','2000'))

df4= df.where(df['ST'].isin('NE','IL'))

print('df.count   :', df.count())
print('df2.count   :', df2.count())
print('df3.count   :', df3.count())
print('df4.count   :', df4.count())

df.count   : 561
df2.count   : 4
df3.count   : 9
df4.count   : 73


# Filter Data Using Logical Operations

In [57]:
df2 = df.where((df['ST'] == 'NE') & (df['City'] == 'Ericson'))
df2.show(5)

+------------------+-------+---+-----+---------------------+------------+
|         Bank Name|   City| ST| CERT|Acquiring Institution|Closing Date|
+------------------+-------+---+-----+---------------------+------------+
|Ericson State Bank|Ericson| NE|18265| Farmers and Merch...|   14-Feb-20|
+------------------+-------+---+-----+---------------------+------------+

